# Module 0, Notebook 2: Building Markov Decision Processes

## Learning Objectives
By completing this notebook, you will:
1. Represent a Markov Decision Process as a Python dictionary with explicit transition probabilities and rewards
2. Compute returns (discounted and undiscounted) for sampled trajectories
3. Understand how the discount factor gamma shapes what the agent cares about
4. Visualize the state-transition graph of a small MDP
5. Sample trajectories from an MDP to build empirical intuition

## Prerequisites
- Notebook 01: agent-environment loop
- Probability basics: distributions, sampling
- Python dictionaries and list comprehensions

## Estimated Time: 15 minutes

---

**Key Insight:** An MDP is just a rigorous way of writing down what the agent-environment loop does. Once you can write an MDP as a dictionary, you can solve it exactly — no simulation required. This is the bridge between "running experiments" and "doing math".

In [ ]:
learning_objectives([
    "Notebook 01: agent-environment loop",
    "Probability basics: distributions, sampling",
    "Python dictionaries and list comprehensions",
    "**Key Insight:** An MDP is just a rigorous way of writing down what the agent-environment loop does. Once you can write an MDP as a dictionary, you can solve it exactly — no simulation required. This is the bridge between "running experiments" and "doing math"."
])

## 1. Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Imports and reproducibility seed
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

np.random.seed(42)
print("Ready. NumPy version:", np.__version__)

## 2. What is a Markov Decision Process?

A **Markov Decision Process** (MDP) is a tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:

| Symbol | Name | Meaning |
|--------|------|---------|
| $\mathcal{S}$ | State space | All possible situations the agent can be in |
| $\mathcal{A}$ | Action space | All choices available to the agent |
| $P(s' \mid s, a)$ | Transition function | Probability of reaching $s'$ from $s$ after action $a$ |
| $R(s, a, s')$ | Reward function | Immediate reward for transitioning $s \xrightarrow{a} s'$ |
| $\gamma \in [0,1)$ | Discount factor | How much the agent values future rewards |

The **Markov property** says the next state depends only on the current state and action — not on the full history. This is what makes MDPs tractable.

We represent MDPs as nested Python dictionaries because dictionaries make the structure explicit and readable.

## 3. Defining an MDP as a Python Dictionary

We build a small 5-state MDP representing a job search problem:
- States: `unemployed`, `searching`, `interviewing`, `employed`, `quit`
- The agent decides whether to search aggressively, apply selectively, or rest

The transition dictionary has structure:
```python
P[state][action] = [(probability, next_state, reward), ...]
```
Each list contains all possible outcomes (probability, next_state, reward). Probabilities for each (state, action) pair must sum to 1.

In [ ]:
# Purpose: Define a 5-state job-search MDP as a dictionary
# Key Concept: P[s][a] = [(prob, next_s, reward), ...] is the standard sparse MDP format

# State and action identifiers
STATES  = ['unemployed', 'searching', 'interviewing', 'employed', 'quit']
ACTIONS = ['search_aggressively', 'apply_selectively', 'rest']

# Convenience indices
S = {name: i for i, name in enumerate(STATES)}
A = {name: i for i, name in enumerate(ACTIONS)}

# MDP transition/reward table
# Format: P[state_index][action_index] = [(prob, next_state_index, reward), ...]
P = {
    # --- unemployed ---
    S['unemployed']: {
        A['search_aggressively']: [
            (0.6, S['searching'],    -1.0),   # cost of active search
            (0.4, S['unemployed'],   -2.0),   # no leads found
        ],
        A['apply_selectively']: [
            (0.3, S['searching'],    -0.5),
            (0.7, S['unemployed'],   -1.0),
        ],
        A['rest']: [
            (1.0, S['unemployed'],   -3.0),   # no income, time passing
        ],
    },
    # --- searching ---
    S['searching']: {
        A['search_aggressively']: [
            (0.5, S['interviewing'],  0.0),
            (0.3, S['searching'],    -1.0),
            (0.2, S['unemployed'],   -2.0),   # lead fell through
        ],
        A['apply_selectively']: [
            (0.4, S['interviewing'],  0.0),
            (0.6, S['searching'],    -0.5),
        ],
        A['rest']: [
            (0.1, S['unemployed'],   -1.0),
            (0.9, S['searching'],    -0.5),
        ],
    },
    # --- interviewing ---
    S['interviewing']: {
        A['search_aggressively']: [
            (0.7, S['employed'],     10.0),   # got the job
            (0.3, S['unemployed'],   -1.0),   # rejected
        ],
        A['apply_selectively']: [
            (0.5, S['employed'],     10.0),
            (0.5, S['unemployed'],   -1.0),
        ],
        A['rest']: [
            (0.2, S['employed'],      8.0),
            (0.8, S['unemployed'],   -2.0),   # unprepared
        ],
    },
    # --- employed (absorbing, but agent can quit) ---
    S['employed']: {
        A['search_aggressively']: [
            (1.0, S['employed'],      5.0),   # steady income
        ],
        A['apply_selectively']: [
            (1.0, S['employed'],      5.0),
        ],
        A['rest']: [
            (0.9, S['employed'],      5.0),
            (0.1, S['quit'],         -5.0),   # burnout / quit
        ],
    },
    # --- quit (absorbing terminal state) ---
    S['quit']: {
        A['search_aggressively']: [(1.0, S['quit'], 0.0)],
        A['apply_selectively']:   [(1.0, S['quit'], 0.0)],
        A['rest']:                [(1.0, S['quit'], 0.0)],
    },
}

# Sanity check: probabilities in each (s,a) must sum to 1
for s in P:
    for a in P[s]:
        total = sum(prob for prob, _, _ in P[s][a])
        assert abs(total - 1.0) < 1e-9, f"Probabilities for (s={s},a={a}) sum to {total}"

print(f"MDP defined: {len(STATES)} states, {len(ACTIONS)} actions")
print(f"Transition entries verified (all prob sums = 1.0)")

## 4. Sampling Trajectories from the MDP

A **trajectory** is a sequence of (state, action, reward) tuples generated by following a policy through the MDP. We sample trajectories using `np.random.choice` weighted by transition probabilities.

The policy here is a dictionary mapping each state to an action index.

In [ ]:
# Purpose: Sample trajectories from the MDP under a given policy
# Key Concept: Trajectories are the raw data from which all RL algorithms learn

def sample_trajectory(P, policy, start_state, max_steps=20, terminal_states=None):
    """
    Sample one trajectory from the MDP.

    Parameters
    ----------
    P : dict
        MDP transition/reward table.
    policy : dict
        Maps state index to action index.
    start_state : int
        Initial state index.
    max_steps : int
        Maximum trajectory length.
    terminal_states : set, optional
        States where the episode ends.

    Returns
    -------
    trajectory : list of (state, action, reward)
    """
    if terminal_states is None:
        terminal_states = set()

    trajectory = []
    state = start_state

    for _ in range(max_steps):
        if state in terminal_states:
            break

        action = policy[state]
        outcomes = P[state][action]

        # Sample next state according to transition probabilities
        probs      = [o[0] for o in outcomes]
        next_states = [o[1] for o in outcomes]
        rewards     = [o[2] for o in outcomes]

        idx = np.random.choice(len(outcomes), p=probs)
        next_state = next_states[idx]
        reward     = rewards[idx]

        trajectory.append((state, action, reward))
        state = next_state

    return trajectory


# Define two contrasting policies
aggressive_policy = {
    S['unemployed']:   A['search_aggressively'],
    S['searching']:    A['search_aggressively'],
    S['interviewing']: A['search_aggressively'],
    S['employed']:     A['rest'],
    S['quit']:         A['rest'],
}

passive_policy = {
    S['unemployed']:   A['rest'],
    S['searching']:    A['rest'],
    S['interviewing']: A['rest'],
    S['employed']:     A['rest'],
    S['quit']:         A['rest'],
}

TERMINAL = {S['quit']}

# Sample one trajectory for each policy
np.random.seed(42)
traj_agg  = sample_trajectory(P, aggressive_policy, S['unemployed'], terminal_states=TERMINAL)
traj_pass = sample_trajectory(P, passive_policy,    S['unemployed'], terminal_states=TERMINAL)

print("Aggressive policy trajectory:")
for t, (s, a, r) in enumerate(traj_agg):
    print(f"  t={t:2d}  state={STATES[s]:15s}  action={ACTIONS[a]:22s}  reward={r:+.1f}")

print("\nPassive policy trajectory:")
for t, (s, a, r) in enumerate(traj_pass):
    print(f"  t={t:2d}  state={STATES[s]:15s}  action={ACTIONS[a]:22s}  reward={r:+.1f}")

## 5. Computing Returns and the Discount Factor

The **return** $G_t$ is the discounted sum of future rewards starting at time $t$:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

The **discount factor** $\gamma$ controls the time horizon:
- $\gamma = 0$: agent cares only about the immediate next reward (myopic)
- $\gamma \to 1$: agent cares equally about all future rewards (far-sighted)
- $\gamma = 0.9$: a reward 10 steps away is worth $0.9^{10} \approx 0.35$ of its face value today

We compute returns using the efficient recursive formula: $G_t = R_{t+1} + \gamma G_{t+1}$ (backwards pass).

In [ ]:
# Purpose: Compute discounted returns from a trajectory
# Key Concept: The backwards pass avoids recomputing sums at each timestep (O(n) vs O(n^2))

def compute_returns(trajectory, gamma=0.99):
    """
    Compute discounted return G_t for each timestep in the trajectory.

    Parameters
    ----------
    trajectory : list of (state, action, reward)
    gamma : float
        Discount factor in [0, 1).

    Returns
    -------
    returns : np.ndarray of shape (len(trajectory),)
        G_t for t = 0, 1, ..., T-1
    """
    rewards = np.array([r for _, _, r in trajectory], dtype=float)
    T = len(rewards)
    returns = np.zeros(T)

    # Backwards accumulation: G_{T-1} = R_T, then G_t = R_{t+1} + gamma * G_{t+1}
    G = 0.0
    for t in range(T - 1, -1, -1):
        G = rewards[t] + gamma * G
        returns[t] = G

    return returns


# Compare returns under different discount factors
gammas = [0.0, 0.5, 0.9, 0.99]
print(f"{'Gamma':>6}  {'G_0 (aggressive)':>18}  {'G_0 (passive)':>16}")
print("-" * 46)
for g in gammas:
    G_agg  = compute_returns(traj_agg,  gamma=g)[0] if traj_agg  else 0.0
    G_pass = compute_returns(traj_pass, gamma=g)[0] if traj_pass else 0.0
    print(f"{g:>6.2f}  {G_agg:>18.3f}  {G_pass:>16.3f}")

## 6. Visualizing the Effect of Gamma

As gamma increases, the agent cares more about distant rewards. This plot shows how the return $G_0$ of our aggressive policy trajectory changes as a function of gamma, and how individual step rewards are discounted over time.

In [ ]:
# Purpose: Visualize how gamma shapes returns and individual reward weights
# Key Concept: Gamma is a hyperparameter that encodes the agent's planning horizon

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: G_0 vs gamma for both policies ---
ax = axes[0]
gamma_range = np.linspace(0, 0.99, 100)

if traj_agg:
    g0_agg  = [compute_returns(traj_agg,  g)[0] for g in gamma_range]
    ax.plot(gamma_range, g0_agg, color='#1c7ed6', linewidth=2.5, label='Aggressive policy')

if traj_pass:
    g0_pass = [compute_returns(traj_pass, g)[0] for g in gamma_range]
    ax.plot(gamma_range, g0_pass, color='#e64980', linewidth=2.5, label='Passive policy')

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Discount factor $\\gamma$', fontsize=12)
ax.set_ylabel('Return $G_0$', fontsize=12)
ax.set_title('Total Return vs. Discount Factor', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Right: Reward weights over time for different gammas ---
ax = axes[1]
T_plot = 15
timesteps = np.arange(T_plot)
colors = ['#2f9e44', '#1c7ed6', '#e64980', '#f76707']

for g, c in zip([0.5, 0.9, 0.95, 0.99], colors):
    weights = g ** timesteps
    ax.plot(timesteps, weights, 'o-', color=c, linewidth=2, markersize=5, label=f'$\\gamma={g}$')

ax.set_xlabel('Timesteps into future', fontsize=12)
ax.set_ylabel('Discount weight $\\gamma^t$', fontsize=12)
ax.set_title('How Much the Agent Values Future Rewards', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig('discount_factor_effects.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to discount_factor_effects.png")

## 7. Visualizing the State Transition Graph

The state transition graph makes the MDP structure visible. Nodes are states; directed edges represent transitions. We show only the dominant transition (highest probability) per (state, action) pair to keep the graph readable.

This visualization reveals which states are reachable, which are absorbing, and which transitions the agent controls.

In [ ]:
# Purpose: Draw the MDP as a directed state-transition graph
# Key Concept: Visualizing structure helps catch errors and builds intuition

def draw_mdp_graph(P, states, actions, ax, policy=None):
    """
    Draw MDP state-transition graph on the given axes.

    Parameters
    ----------
    P : dict
    states : list of str
    actions : list of str
    ax : matplotlib Axes
    policy : dict, optional
        If given, highlight edges corresponding to the policy.
    """
    # Arrange states in a circle
    n = len(states)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
    positions = {i: (np.cos(a), np.sin(a)) for i, a in enumerate(angles)}

    node_colors = ['#74c0fc', '#74c0fc', '#74c0fc', '#51cf66', '#ff6b6b']

    # Draw edges (dominant transition per action, with probability label)
    action_colors = ['#1c7ed6', '#e64980', '#f76707']
    seen_edges = set()

    for s_idx in P:
        x0, y0 = positions[s_idx]
        for a_idx in P[s_idx]:
            outcomes = P[s_idx][a_idx]
            # Pick dominant transition
            dominant = max(outcomes, key=lambda o: o[0])
            prob, s_next, reward = dominant

            edge_key = (s_idx, a_idx, s_next)
            if edge_key in seen_edges or s_idx == s_next:
                continue
            seen_edges.add(edge_key)

            x1, y1 = positions[s_next]
            dx, dy = x1 - x0, y1 - y0
            color = action_colors[a_idx % len(action_colors)]

            # Offset edges slightly for readability
            offset = 0.04 * a_idx
            ax.annotate('',
                xy=(x1 - 0.12 * dx, y1 - 0.12 * dy),
                xytext=(x0 + 0.12 * dx + offset, y0 + 0.12 * dy + offset),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.7)
            )
            # Label with probability
            mid_x = (x0 + x1) / 2 + 0.05
            mid_y = (y0 + y1) / 2 + 0.05
            ax.text(mid_x, mid_y, f'{prob:.1f}', fontsize=7, color=color, alpha=0.8)

    # Draw state nodes
    for i, (name, color) in enumerate(zip(states, node_colors)):
        x, y = positions[i]
        circle = plt.Circle((x, y), 0.13, color=color, zorder=5, linewidth=1.5,
                             edgecolor='#333333')
        ax.add_patch(circle)
        # Wrap long names
        label = name.replace('_', '\n')
        ax.text(x, y, label, ha='center', va='center', fontsize=8,
                fontweight='bold', zorder=6)

    # Legend for actions
    legend_elements = [mpatches.Patch(color=action_colors[i], label=actions[i])
                       for i in range(len(actions))]
    ax.legend(handles=legend_elements, fontsize=8, loc='upper right')

    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('MDP State-Transition Graph\n(edges show dominant transition per action)',
                 fontsize=12, fontweight='bold')


fig, ax = plt.subplots(figsize=(8, 8))
draw_mdp_graph(P, STATES, ACTIONS, ax)
plt.tight_layout()
plt.savefig('mdp_graph.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to mdp_graph.png")

## 8. Monte Carlo Return Estimation

Before we have a full RL algorithm, we can estimate the expected return for each state by sampling many trajectories and averaging. This is the empirical foundation for Monte Carlo methods covered in Module 2.

We run 1,000 trajectories under the aggressive policy and compute the average return from each state.

In [ ]:
# Purpose: Estimate expected return per state via Monte Carlo sampling
# Key Concept: Averaging many sampled returns converges to the true value function

def estimate_state_values(P, policy, start_state, n_trajectories=1000,
                          max_steps=20, gamma=0.99, terminal_states=None):
    """
    Estimate V(s) for each state by averaging first-visit returns.

    Returns
    -------
    V : dict mapping state index to estimated value
    """
    returns_sum   = {s: 0.0 for s in range(len(STATES))}
    returns_count = {s: 0   for s in range(len(STATES))}

    for _ in range(n_trajectories):
        traj = sample_trajectory(P, policy, start_state,
                                  max_steps=max_steps, terminal_states=terminal_states)
        if not traj:
            continue
        returns = compute_returns(traj, gamma=gamma)

        # First-visit: record return only at first occurrence of each state
        visited = set()
        for t, ((state, action, _), G_t) in enumerate(zip(traj, returns)):
            if state not in visited:
                returns_sum[state]   += G_t
                returns_count[state] += 1
                visited.add(state)

    V = {}
    for s in range(len(STATES)):
        if returns_count[s] > 0:
            V[s] = returns_sum[s] / returns_count[s]
        else:
            V[s] = float('nan')
    return V


V_agg  = estimate_state_values(P, aggressive_policy, S['unemployed'],
                                n_trajectories=2000, gamma=0.99, terminal_states=TERMINAL)
V_pass = estimate_state_values(P, passive_policy, S['unemployed'],
                                n_trajectories=2000, gamma=0.99, terminal_states=TERMINAL)

print(f"{'State':>15}  {'V (aggressive)':>16}  {'V (passive)':>12}")
print("-" * 48)
for i, name in enumerate(STATES):
    v_a = f"{V_agg[i]:.3f}"  if not np.isnan(V_agg[i])  else "N/A"
    v_p = f"{V_pass[i]:.3f}" if not np.isnan(V_pass[i]) else "N/A"
    print(f"{name:>15}  {v_a:>16}  {v_p:>12}")

## 9. Exercises

### Exercise 2.1: Add a New State and Action

**Task:** Add a `networking` state to the MDP. From `unemployed`, a new action `attend_event` should transition to `networking` with probability 0.8 (reward=-0.5) and stay `unemployed` with probability 0.2 (reward=-1.0). From `networking`, the agent can `search_aggressively` to reach `interviewing` with probability 0.8 (reward=0.0) or return to `unemployed` with probability 0.2 (reward=-1.0). Define this extended MDP and verify the probability sums.

**Expected Output:** "Extended MDP verified" printed to the console.

**Hints:**
<details>
<summary>Hint 1</summary>
Copy the existing `P` dictionary, add entries for the new state index, and add the new action to existing states where relevant.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
The new state index is `len(STATES)` (i.e., 5). Each entry in `P_extended[5][a]` is a list of `(prob, next_state, reward)` tuples. Don't forget to add the new action to `P_extended[S['unemployed']]`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
STATES_EXTENDED  = STATES + ['networking']
ACTIONS_EXTENDED = ACTIONS + ['attend_event']

S_EXT = {name: i for i, name in enumerate(STATES_EXTENDED)}
A_EXT = {name: i for i, name in enumerate(ACTIONS_EXTENDED)}

# Build P_extended by copying P and adding your new entries
import copy
P_extended = copy.deepcopy(P)

# Step 1: Add 'attend_event' action to 'unemployed' state
# P_extended[S_EXT['unemployed']][A_EXT['attend_event']] = [...]

# Step 2: Add the 'networking' state with its own transitions
# P_extended[S_EXT['networking']] = {...}

# Step 3: Give networking state stub entries for other actions to keep the MDP complete

P_extended_verified = False  # Set to True after your sanity check

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_1():
    networking_idx = S_EXT['networking']
    attend_idx     = A_EXT['attend_event']

    assert networking_idx in P_extended, \
        "Add the 'networking' state (index 5) to P_extended."

    assert attend_idx in P_extended[S_EXT['unemployed']], \
        "'attend_event' action missing from 'unemployed' state."

    # Check probabilities sum to 1
    for s in P_extended:
        for a in P_extended[s]:
            total = sum(prob for prob, _, _ in P_extended[s][a])
            assert abs(total - 1.0) < 1e-9, \
                f"Probabilities for (s={s}, a={a}) sum to {total:.4f}, not 1.0"

    # Check 'unemployed' -> 'attend_event' leads to networking with prob >= 0.7
    outcomes = P_extended[S_EXT['unemployed']][attend_idx]
    net_probs = [p for p, ns, r in outcomes if ns == networking_idx]
    assert net_probs, "'attend_event' from 'unemployed' should sometimes reach 'networking'."
    assert max(net_probs) >= 0.7, \
        f"Transition to 'networking' probability should be >= 0.7, got {max(net_probs)}"

    print("Exercise 2.1 passed! Extended MDP verified.")

test_exercise_2_1()

### Exercise 2.2: Compare Policies by Expected Return

**Task:** Define a `selective_policy` that uses `apply_selectively` in all non-terminal states. Run 2,000 trajectory samples and estimate $V(\text{unemployed})$ for this policy with $\gamma = 0.95$. Compare it against the aggressive policy's estimated value at the same state.

**Expected Output:** Two floats printed: V(unemployed) under each policy.

**Hints:**
<details>
<summary>Hint 1</summary>
Copy the structure of `aggressive_policy` and replace all action values with `A['apply_selectively']`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
selective_policy = None  # Replace with a dict mapping state indices to action indices

V_selective = None  # Replace with estimated state value dict (use estimate_state_values)

v_selective_unemp = None  # V(unemployed) under selective policy
v_agg_unemp       = None  # V(unemployed) under aggressive policy (already computed above)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_2():
    assert selective_policy is not None, "Define selective_policy!"
    assert isinstance(selective_policy, dict), "selective_policy must be a dict."
    assert len(selective_policy) == len(STATES), \
        f"Policy must cover all {len(STATES)} states."

    # All actions in the policy must be valid
    for s, a in selective_policy.items():
        assert a in range(len(ACTIONS)), f"Invalid action {a} for state {s}"

    assert V_selective is not None, "Run estimate_state_values to get V_selective."
    assert v_selective_unemp is not None, "Extract V(unemployed) from V_selective."
    assert v_agg_unemp is not None, "Extract V(unemployed) from V_agg."

    assert isinstance(v_selective_unemp, float), "v_selective_unemp should be a float."

    print(f"Exercise 2.2 passed!")
    print(f"  V(unemployed) selective:  {v_selective_unemp:.3f}")
    print(f"  V(unemployed) aggressive: {v_agg_unemp:.3f}")
    winner = 'aggressive' if v_agg_unemp > v_selective_unemp else 'selective'
    print(f"  Better policy (higher V): {winner}")

test_exercise_2_2()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **An MDP is a dictionary**: `P[state][action] = [(prob, next_state, reward), ...]`. This sparse format scales to large state spaces.
2. **The Markov property** means the next state depends only on the current state and action — history does not matter. This makes MDPs both tractable and testable.
3. **Returns are discounted sums of future rewards**. The backwards accumulation formula $G_t = R_{t+1} + \gamma G_{t+1}$ is efficient and forms the backbone of almost every RL update rule.
4. **Gamma controls the planning horizon**. Low gamma = myopic agent; high gamma = long-term planner. The right value depends on your problem's time scale.
5. **Monte Carlo estimation** of $V(s)$ just means: sample many trajectories starting from $s$ and average the returns. This simple idea underlies Module 2's algorithms.
6. **Visualizing the transition graph** is the fastest way to catch bugs in your MDP definition.

## What's Next

In Notebook 3 (`03_bellman_equations_lab.ipynb`), we move from sampling to exact computation. The Bellman equations let us express $V(s)$ as a system of linear equations and solve it analytically for small MDPs — no simulation required. We'll implement this from scratch and verify it matches your Monte Carlo estimates.

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])